<a href="https://colab.research.google.com/github/keyvanmahmoudi/Machine_Learning_Course/blob/main/solutions/Session13_association_rules_and_anomaly_detection_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۱۳: قوانین انجمنی و تشخیص ناهنجاری

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session13_association_rules_and_anomaly_detection.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules
from sklearn.datasets import make_blobs
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score
import seaborn as sns

sns.set_theme(style="whitegrid")

DATA_URL = "https://raw.githubusercontent.com/keyvanmahmoudi/Machine_Learning_Course/main/data/"
raw = pd.read_csv(DATA_URL + "Market_Basket_Optimisation.csv", header=None)
transactions = [row.dropna().str.strip().tolist() for _, row in raw.iterrows()]
encoder = TransactionEncoder()
basket = pd.DataFrame(encoder.fit_transform(transactions), columns=encoder.columns_)

def to_text(itemset):
    return ", ".join(sorted(itemset))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

قوانین انجمنی را با `min_support=0.005` استخراج کنید. چند قانون با Lift بیشتر از ۳ پیدا می‌شود؟ آیا معنادارند؟

</div>

In [2]:
itemsets = fpgrowth(basket, min_support=0.005, use_colnames=True)
rules = association_rules(itemsets, metric="lift", min_threshold=1.0)
strong = rules[rules["lift"] > 3].copy()
strong["antecedents"] = strong["antecedents"].apply(to_text)
strong["consequents"] = strong["consequents"].apply(to_text)
print(f"Rules with min_support=0.005: {len(rules)}, of which lift > 3: {len(strong)}")
strong.sort_values("lift", ascending=False)[["antecedents", "consequents", "support", "confidence", "lift"]].head(10).round(3)

Rules with min_support=0.005: 1832, of which lift > 3: 50


,antecedents,consequents,support,confidence,lift
1330,pasta,escalope,0.006,0.373,4.701
1331,escalope,pasta,0.006,0.074,4.701
1328,shrimp,pasta,0.005,0.071,4.507
1329,pasta,shrimp,0.005,0.322,4.507
818,whole wheat pasta,olive oil,0.008,0.271,4.122
819,olive oil,whole wheat pasta,0.008,0.121,4.122
1675,ground beef,"herb & pepper, spaghetti",0.006,0.065,4.004
1670,"herb & pepper, spaghetti",ground beef,0.006,0.393,4.004
1669,ground beef,"herb & pepper, mineral water",0.007,0.068,3.976
1664,"herb & pepper, mineral water",ground beef,0.007,0.391,3.976


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

با آستانهٔ Support کمتر، قوانین بسیار بیشتری پیدا می‌شوند و بعضی Lift بالای ۳ دارند. بعضی معنادارند (مثلاً پاستا با اسکالوپ یا میگو، و پاستای سبوس‌دار با روغن زیتون؛ مواد یک غذا)، ولی بسیاری از آن‌ها Support بسیار کمی دارند؛ یعنی فقط در چند ده تراکنش دیده شده‌اند و ممکن است تصادفی باشند. **Lift بالا با Support پایین قابل اعتماد نیست**؛ قبل از اقدام، باید روی دادهٔ دورهٔ دیگری هم بررسی شود.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

قوانینی را پیدا کنید که «mineral water» تالی آن‌ها است. چرا بسیاری از قوانین این داده به آب معدنی ختم می‌شوند؟ (راهنمایی: Support آب معدنی را ببینید.)

</div>

In [3]:
itemsets = fpgrowth(basket, min_support=0.01, use_colnames=True)
rules = association_rules(itemsets, metric="lift", min_threshold=1.0)
rules["consequents"] = rules["consequents"].apply(to_text)
rules["antecedents"] = rules["antecedents"].apply(to_text)
water = rules[rules["consequents"] == "mineral water"]
print(f"Rules ending in 'mineral water': {len(water)} of {len(rules)}")
print(f"Support of mineral water: {basket['mineral water'].mean():.1%}")
water.sort_values("confidence", ascending=False)[["antecedents", "support", "confidence", "lift"]].head(8).round(3)

Rules ending in 'mineral water': 42 of 406
Support of mineral water: 23.8%


,antecedents,support,confidence,lift
365,"eggs, ground beef",0.010,0.507,2.126
354,"ground beef, milk",0.011,0.503,2.110
359,"chocolate, ground beef",0.011,0.474,1.988
217,"frozen vegetables, milk",0.011,0.469,1.967
181,soup,0.023,0.456,1.915
317,"pancakes, spaghetti",0.011,0.455,1.909
57,"olive oil, spaghetti",0.010,0.448,1.878
136,"milk, spaghetti",0.016,0.444,1.861


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

آب معدنی پرفروش‌ترین کالای فروشگاه است و در حدود یک‌چهارم همهٔ خریدها وجود دارد. پس تقریباً هر کالایی که خریده شود، با احتمال قابل توجهی آب معدنی هم در سبد هست و قوانین «X ⇐ آب معدنی» به‌راحتی **Confidence** بالایی پیدا می‌کنند؛ حدود ۴۲ قانون از ۴۰۶ قانون به آب معدنی ختم می‌شوند. Confidence حدود ۵۰٪ برای این قوانین چشمگیر به نظر می‌رسد، ولی Lift حدود ۲ نشان می‌دهد که نیمی از این Confidence فقط از محبوبیت آب معدنی می‌آید (۲۴٪ همهٔ مشتریان، بدون هیچ شرطی، آب معدنی می‌خرند). این دقیقاً همان دلیلی است که نباید فقط به Confidence تکیه کرد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

در Isolation Forest، `contamination` را ۰٫۰۲، ۰٫۰۶ و ۰٫۱۵ قرار دهید و Precision و Recall را مقایسه کنید.

</div>

In [4]:
rng = np.random.default_rng(0)
X_normal, _ = make_blobs(n_samples=470, centers=[[0, 0], [5, 5]], cluster_std=[0.8, 1.6], random_state=42)
X_anomaly = rng.uniform(-5, 11, size=(30, 2))
X_all = np.vstack([X_normal, X_anomaly])
is_anomaly = np.r_[np.zeros(470), np.ones(30)].astype(int)

rows = []
for contamination in [0.02, 0.06, 0.15]:
    flagged = (IsolationForest(contamination=contamination, random_state=42).fit_predict(X_all) == -1).astype(int)
    rows.append({"contamination": contamination, "flagged": flagged.sum(),
                 "precision": precision_score(is_anomaly, flagged), "recall": recall_score(is_anomaly, flagged)})
pd.DataFrame(rows).round(3)

,contamination,flagged,precision,recall
0,0.02,10,1.000,0.333
1,0.06,30,0.633,0.633
2,0.15,75,0.347,0.867


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

`contamination` همان آستانهٔ تصمیم است: نسبت نمونه‌هایی که ناهنجار اعلام می‌شوند.
- مقدار کم (۰٫۰۲): فقط مطمئن‌ترین موارد اعلام می‌شوند؛ Precision بالا ولی Recall پایین.
- مقدار زیاد (۰٫۱۵): بیشتر ناهنجاری‌ها پیدا می‌شوند ولی هشدارهای کاذب هم زیاد می‌شود.

همان مصالحهٔ Precision و Recall جلسهٔ ۵. در عمل `contamination` را بر اساس ظرفیت بررسی کارشناسان (مثلاً «روزی ۵۰ مورد را می‌توانیم بررسی کنیم») تعیین می‌کنند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

روی دادهٔ ریزش مشتری جلسهٔ ۱۰، Isolation Forest را روی ویژگی‌های عددی اجرا کنید و ۲۰ مشتری غیرعادی‌تر را بررسی کنید. چه چیزی آن‌ها را غیرعادی کرده است؟

</div>

In [5]:
churn = pd.read_csv(DATA_URL + "Churn_Modelling.csv")
numeric = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary"]
iso = IsolationForest(random_state=42).fit(churn[numeric])
churn["anomaly_score"] = -iso.score_samples(churn[numeric])

top20 = churn.sort_values("anomaly_score", ascending=False).head(20)
print("Median of all customers:")
print(churn[numeric].median().round(0).to_dict())
print(f"Churn rate among the 20 most anomalous customers: {top20['Exited'].mean():.0%} (overall: {churn['Exited'].mean():.0%})")
top20[numeric + ["Exited", "anomaly_score"]].round(2)

Median of all customers:
{'CreditScore': 652.0, 'Age': 37.0, 'Tenure': 5.0, 'Balance': 97199.0, 'NumOfProducts': 1.0, 'EstimatedSalary': 100194.0}
Churn rate among the 20 most anomalous customers: 90% (overall: 20%)


,CreditScore,Age,Tenure,Balance,NumOfProducts,EstimatedSalary,Exited,anomaly_score
4166,850,45,5,174088.30,4,5669.31,1,0.67
1701,792,50,9,0.00,4,194700.81,1,0.65
9292,461,74,1,186445.31,2,196767.83,0,0.65
5235,645,68,9,0.00,4,176353.87,1,0.65
4606,797,55,10,0.00,4,49418.87,1,0.65
4463,850,70,1,96947.58,3,62282.99,1,0.65
8683,697,49,7,195238.29,4,131083.56,1,0.64
2872,439,52,3,96196.24,4,198874.52,1,0.64
2154,606,65,10,126306.64,3,7861.68,1,0.64
9255,847,41,3,101543.51,4,16025.17,1,0.63


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

مشتریان غیرعادی معمولاً ترکیبی از مقادیر **حدی** دارند: سن بسیار بالا، تعداد محصول ۳ یا ۴ (که خیلی نادر است)، امتیاز اعتباری بسیار پایین یا موجودی بسیار زیاد. یک نکتهٔ جالب: نرخ ریزش در این گروه بسیار بیشتر از میانگین ۲۰٪ است؛ مشتریان با ۳ یا ۴ محصول تقریباً همه ریزش کرده‌اند. تشخیص ناهنجاری بدون استفاده از برچسب، گروهی را پیدا کرده که ارزش بررسی جداگانه دارد.

</div>